# ANÁLISIS DE EVASIÓN DE CLIENTES - TELECOM X (MODELO PREDICTIVO)

## Paquetería de importación.

In [ ]:
import requests
import pandas as pd

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import plotly.express as px

## Extración de datos

In [ ]:
# Estración de datos por url.
url = 'https://raw.githubusercontent.com/BjorkMagno/Portafolio/refs/heads/main/data-science/desafios/TelecomX_Modelo/TelecomX_DataTreate.json'
response = requests.get(url)

data = response.json()

In [ ]:
# Datos en Data Frame.
df = pd.DataFrame(data)
df = pd.json_normalize(data)

df.head()

## Exploración de datos.

In [ ]:
# Cantidad de filas y columnas
df.shape

In [ ]:
# Columnas del Data Frame.
df.columns

In [ ]:
# Información general del Data Frame
df.info()

In [ ]:
# Verificación de valores únicos en cada columna
print(f'{"*"*12}  VALORES ÚNICOS SEGUN LA COLUMNA   {"*"* 12} \n{"-" * 60}')
for column in df.columns:
    unique_values = df[column].unique()
    print(f'Valores únicos en la columna "{column.lower()}": {len(unique_values)}')
    if len(unique_values) < 50:
      print(unique_values)
    print('-' * 60)

In [ ]:
# Prueba de datos nulos
df.isnull().sum()

In [ ]:
# Copia de trabajo.
datos = df.copy()

## Tratamiento de datos.

### Eliminación de columnas irrelevantes.

Eliminación de columnas sin valor predictivo, redudantes, y con posibles causas de fuga.

In [ ]:
# Eliminación de columna customer_id
datos.drop('customer_id', axis=1, inplace=True)
datos.columns

### Estandarización de datos.

In [ ]:
# Tratamiento de las palabras del documento.
datos['account_paymentmethod'] = datos['account_paymentmethod'].str.replace('(automatic)', 'automatic').str.replace(' ','_').str.lower()
datos['account_paymentmethod'].unique()

In [ ]:
datos['account_contract'] = datos['account_contract'].str.replace(' ','_').str.lower()
datos['account_contract'].unique()

In [ ]:
datos['internet_internetservice'] = datos['internet_internetservice'].str.replace(' ','_').str.lower()
datos['internet_internetservice'].unique()

### Agrupación de datos.

In [ ]:
# Copia de para datos agrupados.
datos_agrupados = datos.copy()

In [ ]:
# Columnas de internet.
columnas_internet = ['internet_onlinesecurity', 'internet_onlinebackup',
                     'internet_deviceprotection', 'internet_techsupport',
                     'internet_streamingtv', 'internet_streamingmovies']

In [ ]:
# Columnas estaderizadas y agrupadas.
for columna in columnas_internet:
    datos_agrupados[columna] = datos_agrupados[columna].apply(lambda x: True if x == 'Yes' else False)

In [ ]:
# Estaderización y agrupación de los datos de la columna 'phone_multiplelines'
datos_agrupados['phone_multiplelines'] = datos_agrupados['phone_multiplelines'].apply(lambda x: True if x == 'Yes' else False)

In [ ]:
datos_agrupados.head()

### Tratamiento de datos nulos.

Existen 11 filas nulas al ser un número pequeño de datos optamos por eliminarlos, en caso de ser una cantidad considerabale los remplazariamos por 0.

In [ ]:
# Elimina filas con valores nulos en las columnas especificadas
datos_agrupados = datos_agrupados.dropna(subset=['account_charges_total'])

In [ ]:
# Verifica valores nulos en las dos columnas
print(datos_agrupados['account_charges_total'].isnull().sum())

## Enconding (sin dummy trap)

### Variables caracteristicas y númericas.

In [ ]:
# Variables caracteristicas.
columnas_categorias = ['churn', 'customer_gender', 'customer_seniorcitizen',
       'customer_partner', 'customer_dependents',
       'phone_phoneservice', 'phone_multiplelines', 'internet_internetservice',
       'internet_onlinesecurity', 'internet_onlinebackup',
       'internet_deviceprotection', 'internet_techsupport',
       'internet_streamingtv', 'internet_streamingmovies', 'account_contract',
       'account_paperlessbilling', 'account_paymentmethod',
       ]

### Datos Enconding

In [ ]:
datos_enconding = pd.get_dummies(datos_agrupados, columns=columnas_categorias, drop_first=True)
datos_enconding.head()

In [ ]:
# Estandarización de los nombres de columnas.
datos_enconding.columns = datos_enconding.columns.str.replace('1', 'Yes').str.replace('True','Yes').str.lower()

In [ ]:
datos_enconding.columns

### Verificación de la información.

In [ ]:
datos_enconding.info()

## Verificación de la Proporción de Cancelación. (CHURN)

In [ ]:
datos_enconding['churn_yes'].value_counts()

In [ ]:
ax = px.histogram(datos_enconding, x='churn_yes', color='churn_yes', title='Distribución de Churn',
                  color_discrete_map={True: 'green', False: 'red'}, text_auto=True,labels={'churn_yes': 'CHURN'})
ax.update_layout(showlegend=False)
ax.update_layout(title={'x':0.5, 'y':0.9, 'font':{'size': 20, 'family': 'sans-serif'}}, width= 900, height=500)
ax.update_xaxes(tickvals=[False, True],ticktext=['No', 'Yes'])
ax.update_yaxes(title_text='CLIENTES')

ax.show()

## Normalización de datos.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Columnas numéricas
columnas_numericas = datos_enconding.select_dtypes(include=['int64', 'float64']).columns
columnas_numericas

In [ ]:
# Datos Normalizados
scaler = MinMaxScaler()
datos_enconding[columnas_numericas] = scaler.fit_transform(datos_enconding[columnas_numericas])

datos_enconding.head()

## Análisis de Correlación.

### Correlación con todas las variables.

In [ ]:
# Correlación de las variables.
correlacion = datos_enconding.corr()
correlacion

In [ ]:
# Matriz de conrrelación.
fig, ax = plt.subplots(figsize=(20,16))

ax = sns.heatmap(np.round(correlacion, 2), vmax=1, vmin=-1, center=0,
            square=True, linewidths=.5, annot=True, cbar_kws={"shrink": .5})
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, horizontalalignment='right')
ax.set_title('Matriz de Correlación')

plt.show()

### Correlación con las varibles relevantes.

In [ ]:
# Variable objetivo.
target_var = 'churn_yes'

In [ ]:
# Umbral minimo de correlación absoluta para la selecció.
limit = 0.2

In [ ]:
# Filtración de varibles
variables_relevantes = correlacion.index[abs(correlacion[target_var]) >= limit].tolist()
variables_relevantes

In [ ]:
# Introducir la variable objetivo en caso de no encontrarse
if target_var not in variables_relevantes:
    variables_relevantes.append(target_var)

variables_relevantes

In [ ]:
# Matriz de correlacion con varibles relevantes.
corr_filtrada = correlacion.loc[variables_relevantes, variables_relevantes]
corr_filtrada

In [ ]:
# Máscara para ocultar el triángulo superior de la matriz
mascara = np.triu(np.ones_like(corr_filtrada, dtype=bool))

# Grafica de la matriz
plt.figure(figsize=(12,10))
sns.heatmap(
    corr_filtrada,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.7},
    mask=mascara
)
plt.title(f'Heatmap de variables con correlación >= {limit} con "{target_var}"')
plt.show()

## Análisis Dirigido.

In [ ]:
datos_enconding.columns

In [ ]:
# Funcnion para grafica de relacion de variables numericas con la linea de ajuste
def grafica_relacion_numericas (data, columna_x, columna_y):
  fig = px.scatter(data, x=columna_x, y=columna_y, color='churn_yes',
                   labels={False :'red', True: 'blue'},
                   title=f'Relación entre {columna_x.replace("_"," ")} y {columna_y.replace("_"," ")}',
                   trendline='ols'
                   )
  fig.update_layout(title={'x':0.5, 'y':0.9, 'font':{'size': 20, 'family': 'sans-serif'}},
                    xaxis_title=columna_x.replace("_"," ").upper(),
                    yaxis_title=columna_y.replace("_"," ").upper(),
                    width = 900, height= 500)
  return fig

In [ ]:
# Graficas de las varibles numericas y su relación
monthlyVSdiary = grafica_relacion_numericas(datos_enconding, 'account_charges_monthly', 'account_charges_diary')
monthlyVSdiary.show()

In [ ]:
# Graficas de las varibles numericas y su relación
tenureVStotal = grafica_relacion_numericas(datos_enconding, 'customer_tenure', 'account_charges_total')
tenureVStotal.show()

In [ ]:
totalVSmonthly = grafica_relacion_numericas(datos_enconding, 'account_charges_total', 'account_charges_monthly')
totalVSmonthly.show()

In [ ]:
tenureVSmonthly = grafica_relacion_numericas(datos_enconding, 'customer_tenure', 'account_charges_monthly')
tenureVSmonthly.show()

In [ ]:
# Relacion de las variables caracteristicas con churn
for columna in datos_enconding.columns:
  if columna not in np.append(columnas_numericas, 'churn_yes'):
    ax = grafica_relacion_numericas(datos_enconding.corr(), columna, 'churn_yes')
    ax.show()

In [ ]:
# Relacion de las variables númericas con churn
for columna in columnas_numericas:
  ax = px.box(datos_enconding, y='churn_yes', x=columna, color='churn_yes',
            title=f'churn x {columna.replace("_"," ")}', color_discrete_map={ False : 'red', True : 'blue' })

  ax.update_layout(title={'x':0.5, 'y':0.9, 'font':{'size': 20, 'family': 'sans-serif'}}, width= 900, height=500)
  ax.update_yaxes(title_text='CHURN')
  ax.update_xaxes(title_text=columna.replace("_"," ").upper())

  ax.show()

## Análisis de Multicolinealidad.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

#### Variables para el VIF

In [ ]:
#
X = datos_enconding.drop(columns=['churn_yes'])
X_const = add_constant(X)

In [ ]:
# Conversion del tipo de columnas
X_const = X_const.astype(float)

In [ ]:
# Calcular el VIF
vif_data = pd.DataFrame()
vif_data["Feature"] = X_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]

# Mostrar resultado
display(vif_data.sort_values(by='VIF', ascending=False))

#### Analisis del VIF

| Rango de VIF       | Interpretación                |
| ------------------ | ----------------------------- |
| VIF ≈ 1            | Sin multicolinealidad         |
| 1 < VIF < 5        | Baja (aceptable)              |
| 5 ≤ VIF < 10       | Moderada (vigilar)            |
| VIF ≥ 10           | Alta (¡atención!)             |
| VIF = ∞ (infinito) | Multicolinealidad perfecta ⚠️ |

In [ ]:
# VIF ALTA
columnas_vif_alta = vif_data[vif_data['VIF'] >= 10]['Feature'].tolist()
columnas_vif_alta

In [ ]:
# VIF MODERADA
columnas_vif_moderada = vif_data[(vif_data['VIF'] >= 5) & (vif_data['VIF'] < 10)]['Feature'].tolist()
columnas_vif_moderada

In [ ]:
# VIF BAJA
columnas_vif_baja = vif_data[(vif_data['VIF'] >= 1) & (vif_data['VIF'] < 5)]['Feature'].tolist()
columnas_vif_baja

#### Eliminación de columnas con VIF alta

##### Opción 1

Despues de un análisis observamos que

  * Relación entre 'account_charges_diary' y 'account_charges_monthly'
  * Relacion entre 'internet_internetservice_no' y 'internet_internetservice_fibre_optic'
  * Relación entre 'phone_phoneservice_yes' y 'phone_multiplelines_true'



In [ ]:
columas_to_drop = ['account_charges_diary', 'internet_internetservice_no','phone_phoneservice_yes']

In [ ]:
# Copy por seguridad.
X_const_filtro = X_const.copy()
X_const_filtro.drop(columns=columas_to_drop, inplace=True)

X_const_filtro = X_const_filtro.astype(float)

In [ ]:
# Recalcular el VIF
vif_data = pd.DataFrame()
vif_data["Feature"] = X_const_filtro.columns
vif_data["VIF"] = [variance_inflation_factor(X_const_filtro.values, i) for i in range(X_const_filtro.shape[1])]

# Rresultados
display(vif_data.sort_values(by="VIF", ascending=False))

##### Opcion 2

Existe una relación entre 'account_charges_total' y 'account_charges_monthly', que pasa si quitamos este.

In [ ]:
columnas_drop = ['account_charges_total']

In [ ]:
# Copy por seguridad.
X_filtro_final = X_const_filtro.copy()
X_filtro_final.drop(columns=columnas_drop, inplace=True)

X_filtro_final = X_filtro_final.astype(float)

In [ ]:
# Recalcular el VIF
vif_data_final = pd.DataFrame()
vif_data_final["Feature"] = X_filtro_final.columns
vif_data_final["VIF"] = [variance_inflation_factor(X_filtro_final.values, i) for i in range(X_filtro_final.shape[1])]

# Resultados ordenados
display(vif_data_final.sort_values(by="VIF", ascending=False))

## Separación de Datos

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

In [ ]:
# Separar features y target
X = datos_enconding.drop('churn_yes', axis=1)
y = datos_enconding['churn_yes']

In [ ]:
# Dividir entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Normalización de modelo.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Balanceo con SMOTE
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

## Revision con statmodels

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

In [ ]:
X_train_con_constante=sm.add_constant(X_train[['customer_tenure','account_charges_monthly','account_charges_total']])

X_train_con_constante.head()

In [ ]:
modelo_statsmodels = sm.OLS(y_train,X_train_con_constante,hasconst=True).fit()
print(modelo_statsmodels.summary())

## Creación de Modelos

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

#### Regresión Logistica.

In [ ]:
# Instanciar y entrenar
lr = LogisticRegression(random_state=42)
lr.fit(X_train_bal, y_train_bal)

# Predicciones
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

#### Random Forest

In [ ]:
# Instanciar y entrenar
rf = RandomForestClassifier(random_state=42, max_depth=30)
rf.fit(X_train_bal, y_train_bal)

# Predicciones
y_pred_rf = rf.predict(X_test_scaled)
y_prob_rf = rf.predict_proba(X_test_scaled)[:, 1]

## Evaluación de Modelos

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
# Evaluación
print(f'{"*"*17} Regresión Logística {"*"*17}\n{"-"*55}')
print(f'{" "*7}Exactitud: {accuracy_score(y_test, y_pred_lr)}')
print(f'{" "*7}ROC AUC:   {roc_auc_score(y_test, y_prob_lr)}')
print(f'{"_"*55}\n{"*"*15} Reporte de Clasificación {"*"*14}\n{"-"*55}\n{classification_report(y_test, y_pred_lr)}')
print(f'{"_"*55}\n{"*"*17} Matriz de Confusión {"*"*17}\n {confusion_matrix(y_test, y_pred_lr)}\n')

visualizacion = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_test, y_pred_lr), display_labels=lr.classes_)
visualizacion.plot();

In [ ]:
# Evaluación Random Forest
print(f'{"*"*20} Random Forest {"*"*20}\n{"-"*55}')
print(f'{" "*7}Exactitud: {accuracy_score(y_test, y_pred_rf)}')
print(f'{" "*7}ROC AUC:   {roc_auc_score(y_test, y_prob_rf)}')
print(f'{"_"*55}\n{"*"*15} Reporte de Clasificación {"*"*14}\n{"-"*55}\n{classification_report(y_test, y_pred_rf)}')
print(f'{"_"*55}\n{"*"*17} Matriz de Confusión {"*"*17}\n {confusion_matrix(y_test, y_pred_rf)}\n')

visualizacion = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_test, y_pred_rf), display_labels=rf.classes_)
visualizacion.plot();

## Análisis de la variables importantes

In [ ]:
# Features Importances de Random Forest
importances = rf.feature_importances_
rf_feature_importances = pd.DataFrame({'Features':X.columns,'Importances':(importances*100).round(2)}).sort_values('Importances',ascending=False)
rf_feature_importances

In [ ]:
# Features Importances de Random Forest
lr_feature_importances = pd.DataFrame((lr.coef_*100).round(2),columns=X.columns, index=['Importances']).T.sort_values('Importances',ascending=False)
lr_feature_importances

In [ ]:
# Variables relevantes
variables_relevantes_lr_n = [index for index, value in lr_feature_importances['Importances'].items() if value < -20]
variables_relevantes_lr_n

In [ ]:
variables_relevantes_lr_p = [index for index, value in lr_feature_importances['Importances'].items() if value >20]
variables_relevantes_lr_p

In [ ]:
variables_relevantes_lr = np.append(variables_relevantes_lr_n, variables_relevantes_lr_p)
variables_relevantes_lr

In [ ]:
# Variables importancia
variables_relevantes_rf = [rf_feature_importances.loc[index]['Features'] for index, value in rf_feature_importances['Importances'].items() if value> 3]
variables_relevantes_rf

In [ ]:
for columna in variables_relevantes_lr:
  if columna not in np.append(columnas_numericas, 'churn_yes'):
    ax = grafica_relacion_numericas(datos_enconding.corr(), columna, 'churn_yes')
    ax.show()

## Importación del modelo.

In [ ]:
import pickle

try:
  with open('predicion_cancelacion.pkl', 'wb') as archivo:
    pickle.dump(lr, archivo)
  print('Modelo serializado con éxito')
except Exception as e:
  print('Se presentó un error al almacenar el modelo: ',str(e))

In [ ]:
file_path = '/content/predicion_cancelacion.pkl'

try:
    with open(file_path, 'rb') as file:
        model = pickle.load(file)
    print("¡Modelo cargado con éxito!")
except Exception as e:
    print("Ocurrió un error al cargar el modelo:", str(e))

In [ ]:
#obtener un datos al asar
dato = datos_enconding.sample(1)
index = dato.index[0]

nueva_muestra = dato.drop(['churn_yes'], axis=1)
nueva_muestra = nueva_muestra.values[0]

In [ ]:
# Ejemplo de predicción con datos de entrada 'X'
model.predict([nueva_muestra])[0]

In [ ]:
#Buscar un dato por su nombre del index
datos_enconding.loc[index]['churn_yes']

# Infrome

Se realizó un tratamiento de los datos para la creación de dos modelos de predición, obteniendo la evaluación general de los datos. En ambos modelos existe una exatitud de *0.75 o 0.78*, sin embargo, deseamos saber los factores posibles por la cancelación, por lo que nos interesa más la precisón de los las cancelaciones ('False' de churn_yes).

En la evaluación de los modelos *Random Forest y Regresión Logística* tenemos una presición de falsos del **0.86 y 0.91** respectivamente. Por lo tanto, se opta por el estudio me diante el modelo de **Regresión Logística**, no obstante, se analizó las variables relevantes de ambos modelos, calculando un impacto que este por ariba del 10% y 20% según el caso.

En el análisis de las variables notamos que conciden muchas de las variables esto no es de extrañar, entre ambas, las variables numericas son un factor importate en la cancelación, a decir
* 'customer_tenure'
* 'account_charges_monthly'
* 'account_charges_total'

En las variables ceracteristicas, se puede notar que el tipo de contracto es un factor de cancelación, entre más tiempo el contrato existen más cancelación como se observa en las variables
* 'account_contract_one_year'
* 'account_contract_two_year'

Así mismo otros factores son en el servicio de intenet, es decir, no tener servicio de intenet, o no tener fibra optica, su servicio de streamingtv y streamingmovies.

Se recomienda hacer una valoración en el tipo de clientela para *verificar si el precio se puede cambiar a uno más accesible*, en cuyo caso no sea así, **hacer contratos a corto plazo**, **enfocarse en el servicio de fibra optica y promocionar el uso de internet**. Tambien de ser posible ver como **mejorar los servicios de streaming**.